# 🌍 Huấn luyện Mô hình Phát hiện Vật thể Dollar Street với YOLOv8 trên Kaggle GPU

Notebook này được thiết kế **hoàn toàn tự động**:
- Tự động nhận diện cấu trúc dataset trong `/kaggle/input/` (Hỗ trợ cả dataset gốc **Daily objects around the world**, dataset zip, hoặc dataset YOLO).
- Tự động chuyển đổi sang chuẩn YOLO trong vòng 10–15 giây nếu dùng dataset gốc.
- Huấn luyện mô hình YOLOv8s với GPU T4/P100.
- Đánh giá mAP, biểu đồ Loss, Confusion Matrix.
- Xuất mô hình `best.pt` ra thư mục `/kaggle/working/best.pt` để tải về máy tính.


In [ ]:
# 1. Kiểm tra cấu hình GPU và cài đặt Ultralytics
!nvidia-smi

import torch
print(f"CUDA khả dụng: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Tên GPU: {torch.cuda.get_device_name(0)}")

# Cài đặt hoặc cập nhật Ultralytics
!pip install -q ultralytics
import ultralytics
ultralytics.checks()

## 📦 2. Tự động Nhận diện & Chuẩn bị Dataset YOLO
Đoạn mã dưới đây tự động phát hiện mọi cấu trúc dữ liệu trong `/kaggle/input/`:
- Nếu bạn dùng dataset **Daily objects around the world dataset**: Tự động lọc Top 20 nhãn và chuyển sang YOLO.
- Nếu bạn upload file zip hoặc dataset YOLO có sẵn `data.yaml`: Tự động nhận diện và cấu hình.

In [ ]:
import os, glob, json, shutil, random, yaml
from pathlib import Path

WORKING_DIR = "/kaggle/working/dataset"
os.makedirs(WORKING_DIR, exist_ok=True)

print("=== BƯỚC 1: QUÉT THƯ MỤC /kaggle/input ===")
input_items = glob.glob("/kaggle/input/*")
print("Các mục tìm thấy trong /kaggle/input:", input_items)

# Kiểm tra 1: File data.yaml có sẵn
yaml_candidates = glob.glob("/kaggle/input/**/data.yaml", recursive=True)
# Kiểm tra 2: File zip
zip_candidates = glob.glob("/kaggle/input/**/*.zip", recursive=True) + glob.glob("/kaggle/working/*.zip")

if yaml_candidates:
    src_yaml = yaml_candidates[0]
    src_dir = os.path.dirname(src_yaml)
    print(f"✅ Phát hiện dataset YOLO tại: {src_dir}")
    with open(src_yaml, "r", encoding="utf-8") as f:
        data_cfg = yaml.safe_load(f)
    data_cfg["path"] = src_dir
    yaml_path = os.path.join(WORKING_DIR, "data.yaml")
    with open(yaml_path, "w", encoding="utf-8") as f:
        yaml.dump(data_cfg, f, default_flow_style=False)
    print("✅ Cấu hình data.yaml thành công!")

elif zip_candidates:
    import zipfile
    zip_path = zip_candidates[0]
    print(f"📦 Đang giải nén: {zip_path} -> {WORKING_DIR}")
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(WORKING_DIR)
    extracted_yamls = glob.glob(f"{WORKING_DIR}/**/data.yaml", recursive=True)
    yaml_path = extracted_yamls[0]
    with open(yaml_path, "r", encoding="utf-8") as f:
        data_cfg = yaml.safe_load(f)
    data_cfg["path"] = os.path.dirname(yaml_path)
    with open(yaml_path, "w", encoding="utf-8") as f:
        yaml.dump(data_cfg, f, default_flow_style=False)
    print("✅ Giải nén và cấu hình data.yaml thành công!")

else:
    # Trường hợp 3: Dataset gốc Daily objects around the world (Supervisely format)
    print("⚡ Phát hiện Dataset gốc Daily objects around the world (Supervisely)!")
    print("Đang tự động chuyển đổi Top 20 nhãn sang YOLO ngay trên Kaggle...")
    
    # Tìm thư mục chứa các folder Objects__*
    raw_root = None
    for root, dirs, files in os.walk("/kaggle/input"):
        if any(d.startswith("Objects") for d in dirs):
            raw_root = root
            break
    if not raw_root:
        raw_root = input_items[0] if input_items else "/kaggle/input"
    print(f"📁 Thư mục gốc dữ liệu: {raw_root}")
    
    TOP_20_CLASSES = [
        "Phone", "Computer", "Car", "Bike", "Bed", "Toilet", "Shoe",
        "Stove", "Toothbrush", "Cooking pot", "Soap", "Wardrobe",
        "Dish rack", "Kitchen sink", "Plate", "Shower", "Bowl",
        "Cup", "TV", "Hand"
    ]
    CLASS_TO_FOLDERS = {
        "Phone": ["Objects__phones"], "Computer": ["Objects__computers"],
        "Car": ["Objects__cars"], "Bike": ["Objects__bikes"],
        "Bed": ["Objects__beds", "Objects__guest-beds", "Objects__kids-bed"],
        "Toilet": ["Objects__toilets"], "Shoe": ["Objects_everyday-shoes", "Objects__nicest-shoes"],
        "Stove": ["Objects__stoves"], "Toothbrush": ["Objects_toothbrushes"],
        "Cooking pot": ["Objects__cooking-pots"], "Soap": ["Objects__soaps", "Objects__dish-washing-soaps"],
        "Wardrobe": ["Objects__wardrobes"], "Dish rack": ["Objects__dish-racks"],
        "Kitchen sink": ["Objects__kitchen-sinks"], "Plate": ["Objects__plates"],
        "Shower": ["Objects__showers"], "Bowl": ["Objects__bowls"],
        "Cup": ["Objects__cups"], "TV": ["Objects__tvs"], "Hand": ["Objects_hands"]
    }
    
    class_to_id = {c: i for i, c in enumerate(TOP_20_CLASSES)}
    target_set = set(TOP_20_CLASSES)
    
    images_train = os.path.join(WORKING_DIR, "images/train")
    images_val = os.path.join(WORKING_DIR, "images/val")
    labels_train = os.path.join(WORKING_DIR, "labels/train")
    labels_val = os.path.join(WORKING_DIR, "labels/val")
    for p in [images_train, images_val, labels_train, labels_val]:
        os.makedirs(p, exist_ok=True)
    
    MAX_PER_CLASS = 100
    class_count = {c: 0 for c in TOP_20_CLASSES}
    collected = []
    available_dirs = set(os.listdir(raw_root))
    
    for cls in TOP_20_CLASSES:
        folders = CLASS_TO_FOLDERS.get(cls, [])
        for f in folders:
            if f not in available_dirs:
                continue
            img_dir = os.path.join(raw_root, f, "img")
            ann_dir = os.path.join(raw_root, f, "ann")
            if not (os.path.exists(img_dir) and os.path.exists(ann_dir)):
                continue
            
            for afn in os.listdir(ann_dir):
                if not afn.endswith(".json"):
                    continue
                ifn = afn[:-5] if afn.endswith(".json") else afn
                ipath = os.path.join(img_dir, ifn)
                if not os.path.exists(ipath):
                    continue
                
                try:
                    with open(os.path.join(ann_dir, afn), "r", encoding="utf-8") as jf:
                        d = json.load(jf)
                    w = d.get("size", {}).get("width", 0)
                    h = d.get("size", {}).get("height", 0)
                    if w <= 0 or h <= 0:
                        continue
                    
                    valid_objs = []
                    for obj in d.get("objects", []):
                        ct = obj.get("classTitle")
                        pts = obj.get("points", {}).get("exterior", [])
                        if ct in target_set and len(pts) == 2:
                            x1, y1 = pts[0]
                            x2, y2 = pts[1]
                            xmin, xmax = min(x1, x2), max(x1, x2)
                            ymin, ymax = min(y1, y2), max(y1, y2)
                            if xmax > xmin and ymax > ymin:
                                valid_objs.append((ct, xmin, ymin, xmax, ymax))
                    
                    if not valid_objs:
                        continue
                    if class_count[cls] >= MAX_PER_CLASS:
                        continue
                    class_count[cls] += 1
                    collected.append((ipath, w, h, valid_objs))
                except Exception:
                    continue
    
    print(f"Đã thu thập {len(collected)} ảnh đối tượng.")
    random.seed(42)
    random.shuffle(collected)
    
    split_idx = int(len(collected) * 0.8)
    train_data = collected[:split_idx]
    val_data = collected[split_idx:]
    
    def process_split(data_list, img_dst, lbl_dst):
        for ipath, w, h, objs in data_list:
            fname = Path(ipath).name
            stem = Path(ipath).stem
            shutil.copy2(ipath, os.path.join(img_dst, fname))
            lines = []
            for ct, xmin, ymin, xmax, ymax in objs:
                cid = class_to_id[ct]
                xc = max(0.0, min(1.0, ((xmin + xmax) / 2.0) / w))
                yc = max(0.0, min(1.0, ((ymin + ymax) / 2.0) / h))
                bw = max(0.0, min(1.0, (xmax - xmin) / w))
                bh = max(0.0, min(1.0, (ymax - ymin) / h))
                lines.append(f"{cid} {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}\n")
            with open(os.path.join(lbl_dst, f"{stem}.txt"), "w", encoding="utf-8") as f:
                f.writelines(lines)
    
    print("Đang sao chép ảnh & tạo nhãn YOLO...")
    process_split(train_data, images_train, labels_train)
    process_split(val_data, images_val, labels_val)
    
    yaml_path = os.path.join(WORKING_DIR, "data.yaml")
    data_cfg = {
        "path": WORKING_DIR,
        "train": "images/train",
        "val": "images/val",
        "nc": len(TOP_20_CLASSES),
        "names": TOP_20_CLASSES
    }
    with open(yaml_path, "w", encoding="utf-8") as f:
        yaml.dump(data_cfg, f, default_flow_style=False)
    print(f"✅ ĐÃ TẠO DATASET YOLO HOÀN TẤT: {len(train_data)} train, {len(val_data)} val!")

print("\n--- CẤU HÌNH DATA.YAML ---")
print(f"Đường dẫn yaml: {yaml_path}")
print(f"Số lượng lớp (classes): {data_cfg.get('nc')}")
print(f"Danh sách nhãn: {data_cfg.get('names')}")

## 🚀 3. Huấn luyện Mô hình YOLOv8
Sử dụng kiến trúc `yolov8s.pt` (hoặc `yolov8n.pt`) cho tốc độ huấn luyện nhanh và độ chính xác tối ưu trên đồ vật đời sống.

In [ ]:
from ultralytics import YOLO

# Khởi tạo mô hình (pretrained weights từ COCO giúp hội tụ rất nhanh)
model = YOLO("yolov8s.pt")

# Thiết lập các siêu tham số huấn luyện tối ưu
results = model.train(
    data=yaml_path,
    epochs=35,             # 35 epochs là lý tưởng cho dataset ~1500 ảnh
    imgsz=640,             # Kích thước ảnh chuẩn YOLO
    batch=16,              # Batch size phù hợp GPU T4 16GB
    device=0,              # Sử dụng GPU
    workers=2,
    project="/kaggle/working/runs",
    name="dollar_street_yolo",
    save=True,
    patience=10,           # Early stopping nếu không cải thiện sau 10 epochs
    verbose=True
)

## 📊 4. Đánh giá Kết quả Huấn luyện
Hiển thị biểu đồ Loss, Precision, Recall, mAP50 và Confusion Matrix.

In [ ]:
import matplotlib.pyplot as plt
import cv2

run_dir = "/kaggle/working/runs/dollar_street_yolo"

# Hiển thị đồ thị kết quả huấn luyện
results_png = os.path.join(run_dir, "results.png")
if os.path.exists(results_png):
    img = cv2.imread(results_png)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(16, 10))
    plt.imshow(img)
    plt.axis("off")
    plt.title("Đồ thị Huấn luyện & Đánh giá mAP (Dollar Street)", fontsize=16)
    plt.show()

# Hiển thị ma trận nhầm lẫn (Confusion Matrix)
cm_png = os.path.join(run_dir, "confusion_matrix.png")
if os.path.exists(cm_png):
    img = cv2.imread(cm_png)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(12, 10))
    plt.imshow(img)
    plt.axis("off")
    plt.title("Ma trận nhầm lẫn (Confusion Matrix)", fontsize=14)
    plt.show()

## 🔍 5. Kiểm thử Dự đoán trên Tập Validation

In [ ]:
best_model_path = os.path.join(run_dir, "weights", "best.pt")
best_model = YOLO(best_model_path)

# Lấy 4 ảnh ngẫu nhiên từ tập validation
val_images = glob.glob(os.path.join(WORKING_DIR, "images/val/*.jpg"))[:4]
for img_p in val_images:
    res = best_model.predict(source=img_p, conf=0.25, save=False)
    annotated_frame = res[0].plot()
    plt.figure(figsize=(8, 6))
    plt.imshow(cv2.cvtColor(annotated_frame, cv2.COLOR_BGR2RGB))
    plt.axis("off")
    plt.title(f"Dự đoán: {os.path.basename(img_p)}")
    plt.show()

## 💾 6. Xuất Trọng số `best.pt` và ONNX để Tải về
File `best.pt` sẽ được sao chép ra thư mục gốc `/kaggle/working/best.pt` để bạn dễ dàng bấm nút **Download** trên giao diện Kaggle.

In [ ]:
output_best_pt = "/kaggle/working/best.pt"
shutil.copy2(best_model_path, output_best_pt)
print(f"✅ Đã lưu trọng số tốt nhất: {output_best_pt} ({os.path.getsize(output_best_pt)/1024/1024:.2f} MB)")

# Xuất thêm định dạng ONNX phục vụ nhúng ứng dụng tốc độ cao
try:
    best_model.export(format="onnx")
    print("✅ Đã xuất mô hình ONNX!")
except Exception as e:
    print("Bỏ qua ONNX export:", e)

print("\n🎉 CHÚC MỪNG BẠN ĐÃ HUẤN LUYỆN XONG! Hãy tải file best.pt về và nạp vào phần mềm Desktop để trải nghiệm nhận diện thời gian thực!")